In [1]:
import os
import re
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import kruskal
from sklearn.metrics import pairwise_distances
from sklearn.feature_extraction.text import CountVectorizer

from sentence_transformers import SentenceTransformer
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN

from matplotlib import cm, colormaps
from PIL import Image
import seaborn as sns

sns.set_style("whitegrid")

# Reproducibility
random.seed(42)
np.random.seed(42)

# Paths
workspace_root = Path.cwd()
input_csv = workspace_root / "llama-prescreening-enriched.csv"

if not input_csv.exists():
    raise FileNotFoundError(f"Input CSV not found: {input_csv}")

path_res = workspace_root / "result"
path_res.mkdir(parents=True, exist_ok=True)
path_res_topic_mod_bertopic = path_res

# Load and preprocess abstracts
expected_columns = [
    "id", "doi", "title", "authors", "year",
    "keywords", "venue", "abstract", "included",
    "screening_notes", "prompt_type"
]


def load_input_csv(csv_path: Path) -> pd.DataFrame:
    df_in = pd.read_csv(csv_path)
    df_in.columns = [str(col).strip().lower() for col in df_in.columns]

    # If the file has no header row, load it again with explicit names.
    if "abstract" not in df_in.columns:
        df_in = pd.read_csv(csv_path, header=None, names=expected_columns)

    df_in.columns = [str(col).strip().lower() for col in df_in.columns]
    return df_in


def clean_abstract(text: str):
    text = re.sub(r"http\S+|www\S+", "", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


custom_stopwords = {"evidencenet", "fakenewsnet", "factify", "politifact", "snopes",
                    "rumoureval", "cnn", "weibo", "facebook", "wikipedia"}
pattern = r'\\b(' + '|'.join(re.escape(word) for word in custom_stopwords) + r')\\b'

common_phrases = ["this paper", "in this paper", "in this study", "this work"]

df = load_input_csv(input_csv)
df = df[df["abstract"].notna()].copy()
df = df[df["abstract"].astype(str).str.strip() != ""].copy()
df = df[df["abstract"].astype(str).str.split().str.len() > 20].copy()

cleaned_abstracts = df["abstract"].apply(clean_abstract)
cleaned_abstracts = cleaned_abstracts.str.replace(pattern, '', case=False, regex=True)
for phrase in common_phrases:
    cleaned_abstracts = cleaned_abstracts.str.replace(phrase, '', case=False, regex=False)

df["abstract_clean"] = cleaned_abstracts.str.strip()
df = df[df["abstract_clean"].str.strip() != ""].copy()
df = df.drop_duplicates(subset="abstract_clean").reset_index(drop=True)
df["document_id"] = df.index

text_series = df["abstract_clean"].copy()

# Embedding models to process
embedding_model_names = [
    "all-MiniLM-L6-v2",
    "all-mpnet-base-v2",
    "distiluse-base-multilingual-cased-v1",
    "paraphrase-MiniLM-L6-v2"
]

print(f"Using input CSV: {input_csv}")
print(f"Documents available after preprocessing: {len(df)}")
print(path_res_topic_mod_bertopic)

ModuleNotFoundError: No module named 'numpy'

In [ ]:
from collections import Counter
from itertools import combinations
import ast

import networkx as nx
from matplotlib.patches import Patch


def parse_author_field(author_value):
    if pd.isna(author_value):
        return []

    text = str(author_value).strip()
    if not text:
        return []

    try:
        parsed = ast.literal_eval(text)
        if isinstance(parsed, list):
            candidates = parsed
        else:
            candidates = [parsed]
    except (ValueError, SyntaxError):
        candidates = [text]

    authors = []
    for candidate in candidates:
        candidate_text = str(candidate).strip().strip('"').strip("'")
        if not candidate_text:
            continue
        if ":::" in candidate_text:
            parts = candidate_text.split(":::")
        elif " and " in candidate_text:
            parts = re.split(r"\s+and\s+", candidate_text)
        elif ";" in candidate_text:
            parts = [part.strip() for part in candidate_text.split(";")]
        elif "|" in candidate_text:
            parts = [part.strip() for part in candidate_text.split("|")]
        else:
            parts = [candidate_text]

        for part in parts:
            cleaned = part.strip().strip('"').strip("'")
            if cleaned:
                authors.append(cleaned)

    return authors


def format_author_label(author_name):
    author_name = re.sub(r"\s+", " ", str(author_name).strip())
    if not author_name:
        return None

    if "," in author_name:
        family, given = [piece.strip() for piece in author_name.split(",", 1)]
        initials = "".join(token[0] for token in re.findall(r"[A-Za-zÀ-ÿ]+", given))
        if initials:
            return f"{family} {initials}."
        return family

    tokens = re.findall(r"[A-Za-zÀ-ÿ]+", author_name)
    if not tokens:
        return None
    if len(tokens) == 1:
        return tokens[0]

    family = tokens[-1]
    initials = "".join(token[0] for token in tokens[:-1])
    return f"{family} {initials}." if initials else family


def build_top50_coauthorship_plot(df_source, output_path):
    author_counts = Counter()
    edge_weights = Counter()

    for author_field in df_source["authors"].dropna():
        authors = [format_author_label(name) for name in parse_author_field(author_field)]
        authors = [name for name in dict.fromkeys(authors) if name]
        if len(authors) < 2:
            continue

        for author in authors:
            author_counts[author] += 1

        for author_a, author_b in combinations(sorted(authors), 2):
            edge_weights[(author_a, author_b)] += 1

    collab_strength = Counter()
    for (author_a, author_b), weight in edge_weights.items():
        collab_strength[author_a] += weight
        collab_strength[author_b] += weight

    top_authors = [author for author, _ in collab_strength.most_common(50)]
    graph = nx.Graph()
    graph.add_nodes_from(top_authors)

    for (author_a, author_b), weight in edge_weights.items():
        if author_a in graph and author_b in graph:
            graph.add_edge(author_a, author_b, weight=weight)

    graph.remove_nodes_from(list(nx.isolates(graph)))

    if graph.number_of_nodes() == 0:
        raise ValueError("No coauthorship graph could be built from the authors column.")

    try:
        communities = list(nx.algorithms.community.louvain_communities(graph, weight="weight", seed=42))
    except Exception:
        communities = list(nx.algorithms.community.greedy_modularity_communities(graph, weight="weight"))

    community_map = {}
    for community_id, community in enumerate(communities):
        for author in community:
            community_map[author] = community_id

    color_cycle = plt.get_cmap("tab20")
    node_colors = [color_cycle(community_map.get(author, 0) % 20) for author in graph.nodes()]
    node_sizes = [900 + 180 * collab_strength.get(author, 1) for author in graph.nodes()]
    edge_widths = [0.4 + 0.45 * graph[u][v].get("weight", 1) for u, v in graph.edges()]

    plt.figure(figsize=(16, 14), facecolor="white")
    positions = nx.spring_layout(graph, seed=42, weight="weight", k=0.9, iterations=300)

    nx.draw_networkx_edges(
        graph,
        positions,
        width=edge_widths,
        alpha=0.28,
        edge_color="#9a9a9a"
    )
    nx.draw_networkx_nodes(
        graph,
        positions,
        node_color=node_colors,
        node_size=node_sizes,
        linewidths=1.0,
        edgecolors="white"
    )
    nx.draw_networkx_labels(
        graph,
        positions,
        font_size=15,
        font_family="sans-serif",
        font_color="black"
    )

    legend_handles = []
    for community_id in range(len(communities)):
        legend_handles.append(
            Patch(
                facecolor=color_cycle(community_id % 20),
                edgecolor="none",
                label=f"Community {community_id}"
            )
        )

    plt.title("Top 50 Authors by Co-authorship (Louvain Communities)", fontsize=26, pad=24)
    plt.legend(
        handles=legend_handles,
        title="Louvain Communities",
        loc="lower left",
        frameon=True,
        framealpha=0.92,
        fontsize=18,
        title_fontsize=18
    )
    plt.axis("off")
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()

    print(f"Saved coauthorship plot to: {output_path}")
    print(f"Nodes: {graph.number_of_nodes()}, edges: {graph.number_of_edges()}, communities: {len(communities)}")


coauthorship_plot_path = workspace_root / "llama-coauthorship.png"
build_top50_coauthorship_plot(df, coauthorship_plot_path)

In [2]:
# Ensure abstract_clean exists
df_docs = df["abstract_clean"].tolist()

# Optional metadata fields to keep when present in the CSV
metadata_fields = [col for col in ["id", "doi", "title", "year"] if col in df.columns]
df_metadata = df[metadata_fields].copy() if metadata_fields else pd.DataFrame(index=df.index)
df_metadata["doc_index"] = df_metadata.index  # for merging later

# Loop over each embedding model
for embedding_model_name in embedding_model_names:
    print(f"\n=== Processing: {embedding_model_name} ===")

    normalized_model_name = embedding_model_name.replace("/", "_")
    path_model_dir = path_res_topic_mod_bertopic / normalized_model_name
    path_model_dir.mkdir(parents=True, exist_ok=True)

    path_model_file = path_model_dir / f"bertopic_model.pkl"
    path_topic_csv = path_model_dir / "1_documents_dataset_mapping.csv"
    path_embeddings = path_model_dir / "documents_embeddings.npy"

    model_exists = path_model_file.exists()
    csv_exists = path_topic_csv.exists()
    embeddings_exist = path_embeddings.exists()

    if model_exists and csv_exists and embeddings_exist:
        print("Loading existing BERTopic model and embeddings...")
        topic_model = BERTopic.load(str(path_model_file))
        df_out = pd.read_csv(path_topic_csv)
        embeddings = np.load(path_embeddings)

    else:
        print("Fitting new BERTopic model...")

        # Encode abstracts
        sentence_model = SentenceTransformer(embedding_model_name)
        embeddings = sentence_model.encode(text_series.tolist(), show_progress_bar=True)
        np.save(path_embeddings, embeddings)

        # Clustering and reduction models
        umap_model = UMAP(n_neighbors=15, n_components=50, metric='cosine', random_state=42)
        hdbscan_model = HDBSCAN(
            min_cluster_size=5,
            min_samples=2,
            metric='euclidean',
            cluster_selection_method='leaf'
        )
        vectorizer_model = CountVectorizer(stop_words="english")

        topic_model = BERTopic(
            embedding_model=None,
            umap_model=umap_model,
            hdbscan_model=hdbscan_model,
            vectorizer_model=vectorizer_model,
            verbose=True
        )

        topics, probs = topic_model.fit_transform(text_series.tolist(), embeddings)

        # Assemble DataFrame with IDs and metadata
        df_out = pd.DataFrame({
            "document_id": list(range(len(text_series))),
            "text": text_series,
            "topic_id": topics,
            "probability": [float(p) if p is not None else None for p in probs]
        })

        # Add original metadata
        df_out = df_out.merge(df_metadata, left_on="document_id", right_on="doc_index", how="left")

        # Save both model and CSV
        topic_model.save(str(path_model_file))
        df_out.to_csv(path_topic_csv, index=False)

    print(f"✓ BERTopic model and document-topic assignments ready for {embedding_model_name}")


=== Processing: all-MiniLM-L6-v2 ===
Fitting new BERTopic model...


Batches: 100%|██████████| 31/31 [00:07<00:00,  4.34it/s]
2026-05-03 21:19:50,986 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-03 21:20:00,046 - BERTopic - Dimensionality - Completed ✓
2026-05-03 21:20:00,047 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-03 21:20:00,080 - BERTopic - Cluster - Completed ✓
2026-05-03 21:20:00,083 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-03 21:20:00,212 - BERTopic - Representation - Completed ✓
2026-05-03 21:20:00,362 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ BERTopic model and document-topic assignments ready for all-MiniLM-L6-v2

=== Processing: all-mpnet-base-v2 ===
Fitting new BERTopic model...


Batches: 100%|██████████| 31/31 [00:48<00:00,  1.57s/it]
2026-05-03 21:20:55,965 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-03 21:20:59,038 - BERTopic - Dimensionality - Completed ✓
2026-05-03 21:20:59,039 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-03 21:20:59,071 - BERTopic - Cluster - Completed ✓
2026-05-03 21:20:59,074 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-03 21:20:59,206 - BERTopic - Representation - Completed ✓
2026-05-03 21:20:59,365 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ BERTopic model and document-topic assignments ready for all-mpnet-base-v2

=== Processing: distiluse-base-multilingual-cased-v1 ===
Fitting new BERTopic model...


Batches: 100%|██████████| 31/31 [00:08<00:00,  3.52it/s]
2026-05-03 21:21:17,414 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-03 21:21:20,410 - BERTopic - Dimensionality - Completed ✓
2026-05-03 21:21:20,411 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-03 21:21:20,449 - BERTopic - Cluster - Completed ✓
2026-05-03 21:21:20,451 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-03 21:21:20,581 - BERTopic - Representation - Completed ✓
2026-05-03 21:21:20,739 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ BERTopic model and document-topic assignments ready for distiluse-base-multilingual-cased-v1

=== Processing: paraphrase-MiniLM-L6-v2 ===
Fitting new BERTopic model...


Batches: 100%|██████████| 31/31 [00:03<00:00,  9.57it/s]
2026-05-03 21:21:30,511 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-05-03 21:21:33,475 - BERTopic - Dimensionality - Completed ✓
2026-05-03 21:21:33,476 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-05-03 21:21:33,513 - BERTopic - Cluster - Completed ✓
2026-05-03 21:21:33,516 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-05-03 21:21:33,647 - BERTopic - Representation - Completed ✓
2026-05-03 21:21:33,811 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


✓ BERTopic model and document-topic assignments ready for paraphrase-MiniLM-L6-v2


In [3]:
for embedding_model_name in embedding_model_names:
    print(f"\n=== Summary for model: {embedding_model_name} ===")

    # Normalize model name for file paths
    normalized_model_name = embedding_model_name.replace("/", "_")
    path_model = path_res_topic_mod_bertopic / normalized_model_name / f"bertopic_model.pkl"

    if path_model.exists():
        try:
            model = BERTopic.load(str(path_model))
            topic_info = model.get_topic_info()
            num_topics = len(topic_info)
            topic_sizes = topic_info["Count"].tolist()

            print(f"Loaded model from: {path_model}")
            print(f"Number of topics: {num_topics}")
            print(f"Topic sizes array: {topic_sizes}")
            print(f"Total docs: {sum(topic_sizes)}")

        except Exception as e:
            print(f"Failed to load model at {path_model}: {e}")
    else:
        print(f"Model not found at {path_model}")


=== Summary for model: all-MiniLM-L6-v2 ===
Loaded model from: /Users/ap4320861gmail.com/Library/Mobile Documents/com~apple~CloudDocs/minor-project/results/topic-modeling/bertopic/result/all-MiniLM-L6-v2/bertopic_model.pkl
Number of topics: 69
Topic sizes array: [363, 25, 22, 21, 17, 17, 17, 16, 15, 15, 15, 15, 15, 14, 14, 13, 12, 11, 11, 11, 11, 10, 10, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]
Total docs: 983

=== Summary for model: all-mpnet-base-v2 ===
Loaded model from: /Users/ap4320861gmail.com/Library/Mobile Documents/com~apple~CloudDocs/minor-project/results/topic-modeling/bertopic/result/all-mpnet-base-v2/bertopic_model.pkl
Number of topics: 67
Topic sizes array: [268, 33, 32, 30, 24, 20, 20, 19, 18, 17, 16, 16, 14, 13, 13, 13, 12, 12, 12, 12, 11, 11, 11, 11, 11, 11, 11, 11, 11, 10, 10, 10, 10, 10, 10, 10, 10, 9, 9, 9, 8, 8, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 

In [4]:
def suggest_topic_name(top_words: list[str]) -> str:
    """
    Suggests a short, human-readable name for a topic based on its top words.
    Uses:
      1) A small dictionary of keyword patterns -> specific domain labels
      2) A fallback heuristic that forms a short phrase from top words
    """
    # Example dictionary for domain-specific patterns:
    # keys: sets of indicative words (lowercased),
    # values: a short topic label
    pattern_dict = [
        ({"fake", "fakenews", "propaganda", "disinformation", "misinformation"},
         "Misinformation & Propaganda"),
        ({"debunking", "factcheck", "veracity", "refuting", "factchecking"},
         "Fact-Checking & Verification"),
        ({"credibility", "trust", "reliable", "confidence"},
         "Credibility Assessment"),
        ({"journalism", "journalistic", "media", "reporting"},
         "Journalism & Media Studies"),
        ({"rumors", "rumor", "rumour", "retweets", "twitter", "microblogs"},
         "Rumor on Social Media"),
        ({"crowdsourcing", "crowd", "annotation", "human"},
         "Crowdsourced Analysis"),
        ({"nlp", "classifiers", "embedding", "bayes", "transformer"},
         "NLP Methods"),
        # ... add more domain-specific mappings
    ]

    # Normalize input (lowercase)
    lower_words = {w.lower() for w in top_words}

    # First pass: check explicit pattern dictionary
    for word_set, label in pattern_dict:
        if lower_words & word_set:  # intersection not empty
            return label

    # Second pass: generate a short “keyword-based” label
    # e.g., combine the first 2-3 significant words
    # filter out too-generic words like 'news', 'paper', 'model', etc.
    generic_terms = {"news", "paper", "model", "dataset", "data", "using",
                     "analysis", "study", "approach", "method", "methods"}
    significant = [w for w in top_words if w.lower() not in generic_terms]
    if len(significant) >= 2:
        return ", ".join(significant[:2])
    elif significant:
        return significant[0]
    else:
        # fallback if all words are generic
        return ", ".join(top_words[:2])


for embedding_model_name in embedding_model_names:
    print(f"\n=== Exporting topics for model: {embedding_model_name} ===")

    normalized_model_name = embedding_model_name.replace("/", "_")
    path_model_dir = path_res_topic_mod_bertopic / normalized_model_name
    path_model_file = path_model_dir / "bertopic_model.pkl"
    path_topic_csv = path_model_dir / "1_documents_dataset_mapping.csv"
    path_topic_dir = path_model_dir / "topics"
    path_topic_dir.mkdir(parents=True, exist_ok=True)

    if not path_model_file.exists() or not path_topic_csv.exists():
        print(f"Missing model or topic CSV for: {embedding_model_name}")
        continue

    model = BERTopic.load(str(path_model_file))
    df_topics = pd.read_csv(path_topic_csv)
    topic_info_df = model.get_topic_info()

    summary_records = []

    for _, row in topic_info_df.iterrows():
        topic_id = int(row["Topic"])

        keywords = model.get_topic(topic_id)
        top_words = [kw for kw, _ in keywords[:10]]

        # Get all documents assigned to this topic
        topic_docs_df = df_topics[df_topics["topic_id"] == topic_id]
        topic_docs = topic_docs_df["text"].tolist()

        topic_data = {
            "topic_id": topic_id,
            "label": f"T{topic_id:03d}",
            "label_full" : suggest_topic_name(top_words),
            "keywords": top_words,
            "n_docs": len(topic_docs),
            "documents": []
        }

        for i, (idx, doc_text) in enumerate(zip(topic_docs_df.index, topic_docs)):
            topic_data["documents"].append({
                "document_id": int(idx),
                "score": None,  # You can add probabilities here if you saved them
                "abstract": "\n".join(
                    [" ".join(doc_text.split()[j:j + 25]) for j in range(0, len(doc_text.split()), 25)]
                )
            })

        # Save per-topic JSON
        topic_file = path_topic_dir / f"topic_{topic_id:03d}.json"
        with open(topic_file, "w", encoding="utf-8") as f:
            json.dump(topic_data, f, ensure_ascii=False, indent=2)

        summary_records.append({
            "topic_id": topic_id,
            "label": f"T{topic_id:03d}",
            "label_full" : suggest_topic_name(top_words),
            "n_docs": len(topic_docs),
            "top_words": ", ".join(top_words)
        })

        print(f"Topic #{topic_id:03d} → {top_words}")

    # Save summary table
    df_summary = pd.DataFrame(summary_records).sort_values("topic_id")
    csv_path = path_model_dir / "2_topic_summary.csv"
    df_summary.to_csv(csv_path, index=False)

    latex_path = path_model_dir / "2_topic_summary.tex"
    df_summary.to_latex(buf=latex_path, index=False, escape=False)

    print(f"✓ Saved {len(summary_records)} topics for {embedding_model_name}")
    print(f"JSONs in:     {path_topic_dir}")
    print(f"Summary CSV:  {csv_path}")
    print(f"LaTeX file:   {latex_path}")


=== Exporting topics for model: all-MiniLM-L6-v2 ===
Topic #-01 → ['news', 'fake', 'information', 'detection', 'model', 'social', 'media', 'content', 'learning', 'veracity']
Topic #000 → ['news', 'headlines', 'participants', 'analytical', 'reasoning', 'false', 'true', 'misinformation', 'vs', 'perceived']
Topic #001 → ['explanations', 'reasoning', 'factchecking', 'fact', 'verification', 'explainable', 'claim', 'human', 'claims', 'systems']
Topic #002 → ['discovery', 'truth', 'sensing', 'sources', 'observations', 'data', 'claims', 'problem', 'factfinding', 'big']
Topic #003 → ['fact', 'checking', 'knowledge', 'multiclaim', 'statement', 'evidence', 'claims', 'category', 'kg', 'verification']
Topic #004 → ['workers', 'crowd', 'crowdsourcing', 'truthfulness', 'crowdsourced', 'statements', 'notes', 'assessments', 'experts', 'judgments']
Topic #005 → ['bert', 'transformers', 'news', 'layer', 'fake', 'lstm', 'deep', 'accuracy', 'recurrent', 'models']
Topic #006 → ['lab', 'task', 'english', 'c

In [5]:
for embedding_model_name in embedding_model_names:
    print(f"\n=== Filtering topics for: {embedding_model_name} ===")

    normalized_model_name = embedding_model_name.replace("/", "_")
    path_model_dir = path_res_topic_mod_bertopic / normalized_model_name
    path_topic_dir = path_model_dir / "topics"
    path_topic_dir.mkdir(parents=True, exist_ok=True)

    topic_name_path = path_model_dir / "topic_names.json"
    if topic_name_path.exists():
        with open(topic_name_path, encoding="utf-8") as f:
            topic_names = {int(k): v for k, v in json.load(f).items()}
    else:
        topic_names = {}

    filtered_docs = []
    excluded_topics = {}

    for file_path in path_topic_dir.glob("*.json"):
        with file_path.open("r", encoding="utf-8") as f:
            topic_data = json.load(f)

        topic_id = topic_data.get("topic_id")
        top_words = topic_data.get("keywords", [])
        topic_label = topic_names.get(topic_id) or suggest_topic_name(top_words)

        documents = topic_data.get("documents", [])
        if topic_label:
            for doc in documents:
                filtered_docs.append({
                    "embedding_model": embedding_model_name,
                    "topic_id": topic_id,
                    "topic_label": topic_label,
                    "document_id": doc["document_id"],
                    "score": doc.get("score")
                })
        else:
            excluded_topics[topic_id] = top_words
            print(f"Excluded topic_id={topic_id} from {file_path.name}")

    df_filtered = pd.DataFrame(filtered_docs)
    df_filtered.sort_values(by=["topic_id", "score"], ascending=[True, False], inplace=True, na_position='last')

    df_filtered.to_json(path_model_dir / "3_documents_filtered_topic.json", orient="records", force_ascii=False, indent=2)
    df_filtered.to_csv(path_model_dir / "3_documents_filtered_topic.csv", index=False)

    with (path_model_dir / "4_excluded_topics.json").open("w", encoding="utf-8") as f:
        json.dump(dict(sorted(excluded_topics.items())), f, indent=2, ensure_ascii=False)

    print(f"Saved filtered documents for {embedding_model_name}")
    print(f"Included topics: {df_filtered['topic_id'].nunique()}")
    print(f"Excluded topics: {len(excluded_topics)}")


=== Filtering topics for: all-MiniLM-L6-v2 ===
Saved filtered documents for all-MiniLM-L6-v2
Included topics: 69
Excluded topics: 0

=== Filtering topics for: all-mpnet-base-v2 ===
Saved filtered documents for all-mpnet-base-v2
Included topics: 67
Excluded topics: 0

=== Filtering topics for: distiluse-base-multilingual-cased-v1 ===
Saved filtered documents for distiluse-base-multilingual-cased-v1
Included topics: 65
Excluded topics: 0

=== Filtering topics for: paraphrase-MiniLM-L6-v2 ===
Saved filtered documents for paraphrase-MiniLM-L6-v2
Included topics: 67
Excluded topics: 0


In [6]:
cohesion_percentile = 75  # Use 50 or 25 depending on desired strictness

all_model_densities_bertopic = {}
model_percentile_thresholds_bertopic = {}

# Global UMAP limits to standardize axes (still stored, but not used for plotting now)
shared_umap_limits = {
    "x_min": float("inf"), "x_max": float("-inf"),
    "y_min": float("inf"), "y_max": float("-inf")
}

umap_results = {}

for embedding_model_name in embedding_model_names:
    print(f"\n=== Generating UMAP plots for: {embedding_model_name} ===")

    normalized_model_name = embedding_model_name.replace("/", "_")
    path_model_dir = path_res_topic_mod_bertopic / normalized_model_name
    path_figures = path_model_dir / "figures"
    path_figures.mkdir(parents=True, exist_ok=True)

    path_filtered = path_model_dir / "3_documents_filtered_topic.csv"
    path_model_file = path_model_dir / "bertopic_model.pkl"
    path_embeddings = path_model_dir / "documents_embeddings.npy"

    if not (path_filtered.exists() and path_model_file.exists() and path_embeddings.exists()):
        print(f"Skipping {embedding_model_name} — missing required files.")
        continue

    df_filtered = pd.read_csv(path_filtered)
    df_filtered["document_id"] = df_filtered["document_id"].astype(int)
    df_filtered["topic_id"] = df_filtered["topic_id"].astype(int)
    # TODO: df_filtered = df_filtered[df_filtered["topic_id"] >= 0]
    # Check this carefully in BERTopic outputs

    topic_model = BERTopic.load(path_model_file)
    all_embeddings = np.load(path_embeddings)

    try:
        filtered_embeddings = np.array([all_embeddings[i] for i in df_filtered["document_id"]])
    except IndexError as e:
        print(f"IndexError: {e} — skipping {embedding_model_name}")
        continue

    reducer = UMAP(n_components=2, random_state=42)
    embedding_2d = reducer.fit_transform(filtered_embeddings)

    # Update shared global limits (for logging/reference, not used in plotting anymore)
    shared_umap_limits["x_min"] = min(shared_umap_limits["x_min"], embedding_2d[:, 0].min())
    shared_umap_limits["x_max"] = max(shared_umap_limits["x_max"], embedding_2d[:, 0].max())
    shared_umap_limits["y_min"] = min(shared_umap_limits["y_min"], embedding_2d[:, 1].min())
    shared_umap_limits["y_max"] = max(shared_umap_limits["y_max"], embedding_2d[:, 1].max())

    topic_densities = {}
    unique_topics = sorted(df_filtered["topic_id"].unique())
    all_scores = []

    for topic_id in unique_topics:
        indices = df_filtered[df_filtered["topic_id"] == topic_id].index
        coords = embedding_2d[indices]
        if len(coords) > 1:
            dist_matrix = pairwise_distances(coords)
            mean_dist = dist_matrix[np.triu_indices_from(dist_matrix, k=1)].mean()
        else:
            mean_dist = 0
        topic_densities[topic_id] = float(mean_dist)
        all_scores.append(mean_dist)

    all_model_densities_bertopic[embedding_model_name] = all_scores
    cohesion_threshold = np.percentile(all_scores, cohesion_percentile)
    model_percentile_thresholds_bertopic[embedding_model_name] = cohesion_threshold

    cmap = colormaps["tab20"]
    topic_to_color = {tid: cmap(i % 20) for i, tid in enumerate(unique_topics)}

    df_filtered["x"] = embedding_2d[:, 0]
    df_filtered["y"] = embedding_2d[:, 1]
    df_filtered["density"] = df_filtered["topic_id"].map(topic_densities)
    df_filtered["color"] = df_filtered["topic_id"].map(topic_to_color)
    df_filtered["label"] = df_filtered.apply(lambda row: f"T{row['topic_id']}\n{row['density']:.2f}", axis=1)
    df_filtered["cohesive"] = df_filtered["density"] < cohesion_threshold

    # Save full filtered DataFrame
    df_filtered.to_csv(path_model_dir / "4_umap_coords.csv", index=False)

    # Save topic densities
    df_cohesion = pd.DataFrame({
        "topic_id": list(topic_densities.keys()),
        "density": list(topic_densities.values())
    })
    df_cohesion.to_csv(path_model_dir / "4_topic_cohesion.csv", index=False)

    # Save histogram of topic cohesion scores
    plt.figure()
    plt.hist(all_scores, bins=30)
    plt.axvline(cohesion_threshold, color="red", linestyle="--", label="Threshold")
    plt.title(f"Topic Cohesion Histogram — {embedding_model_name}")
    plt.xlabel("Mean Intra-topic Distance")
    plt.ylabel("Topic Count")
    plt.legend()
    plt.savefig(path_figures / "topic_cohesion_histogram.png", dpi=300)
    plt.close()

    # Save 2D embeddings and bounding box info
    np.save(path_model_dir / "4_umap_embedding.npy", embedding_2d)
    with open(path_model_dir / "4_umap_bounds.json", "w") as f:
        json.dump({k: float(v) for k, v in shared_umap_limits.items()}, f, indent=2)

    # Store results for second-pass plotting
    umap_results[embedding_model_name] = (df_filtered, embedding_2d, cohesion_threshold, topic_to_color)

# --- Second pass: plot with global and local axes ---
for embedding_model_name, (df_filtered, embedding_2d, cohesion_threshold, topic_to_color) in umap_results.items():
    normalized_model_name = embedding_model_name.replace("/", "_")
    path_model_dir = path_res_topic_mod_bertopic / normalized_model_name
    path_figures = path_model_dir / "figures"

    unique_topics = sorted(df_filtered["topic_id"].unique())
    topic_densities = df_filtered.groupby("topic_id")["density"].first().to_dict()

    # --- Plot 1a: All Topics (Global Axes) ---
    plt.figure(figsize=(6, 4))
    for tid in unique_topics:
        coords = df_filtered[df_filtered["topic_id"] == tid][["x", "y"]].values
        x, y = coords[:, 0], coords[:, 1]
        label = df_filtered[df_filtered["topic_id"] == tid]["topic_label"].iloc[0] if "topic_label" in df_filtered.columns else f"T{tid}"
        plt.scatter(x, y, color=topic_to_color[tid], label=f"T{tid}: {label}", s=20, alpha=0.7)
    plt.xlim(shared_umap_limits["x_min"], shared_umap_limits["x_max"])
    plt.ylim(shared_umap_limits["y_min"], shared_umap_limits["y_max"])
    #plt.title(f"Topics UMAP — {embedding_model_name}", fontsize=13)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(path_figures / "filtered_topics_umap_all_fixed_axes.png", dpi=300, bbox_inches="tight")
    plt.close()

    # --- Plot 1b: All Topics (Local Axes) ---
    plt.figure(figsize=(6, 4))
    for tid in unique_topics:
        coords = df_filtered[df_filtered["topic_id"] == tid][["x", "y"]].values
        x, y = coords[:, 0], coords[:, 1]
        plt.scatter(x, y, color=topic_to_color[tid], s=20, alpha=0.7)
    #plt.title(f"Topics UMAP — {embedding_model_name}", fontsize=13)
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(path_figures / "filtered_topics_umap_all_local_axes.png", dpi=300, bbox_inches="tight")
    plt.close()

    # --- Plot 2a: Cohesive Topics Only (Global Axes) ---
    retained_df = df_filtered[df_filtered["cohesive"]].copy()
    plt.figure(figsize=(6, 4))
    for tid in retained_df["topic_id"].unique():
        group = retained_df[retained_df["topic_id"] == tid]
        x, y = group["x"].values, group["y"].values
        plt.scatter(x, y, color=group["color"].iloc[0], s=20, alpha=0.7)
    plt.xlim(shared_umap_limits["x_min"], shared_umap_limits["x_max"])
    plt.ylim(shared_umap_limits["y_min"], shared_umap_limits["y_max"])
    n_retained = retained_df["topic_id"].nunique()
    n_total = len(topic_densities)
    plt.title(
        f"Below {cohesion_percentile}th Percentile = {cohesion_threshold:.2f}, "
        f"Retained: {retained_df['topic_id'].nunique()} of {len(topic_densities)}", fontsize=14
    )
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(path_figures / "filtered_topics_umap_cohesive_fixed_axes.png", dpi=300, bbox_inches="tight")
    plt.close()

    # --- Plot 2b: Cohesive Topics Only (Local Axes) ---
    plt.figure(figsize=(6, 4))
    for tid in retained_df["topic_id"].unique():
        group = retained_df[retained_df["topic_id"] == tid]
        x, y = group["x"].values, group["y"].values
        plt.scatter(x, y, color=group["color"].iloc[0], s=20, alpha=0.7)
    plt.title(
        f"Below {cohesion_percentile}th Percentile = {cohesion_threshold:.2f}, "
        f"Retained: {retained_df['topic_id'].nunique()} of {len(topic_densities)}", fontsize=14
    )
    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.tight_layout()
    plt.savefig(path_figures / "filtered_topics_umap_cohesive_local_axes.png", dpi=300, bbox_inches="tight")
    plt.close()

    # --- Plot 3: Topic Size CCDF ---
    topic_sizes = df_filtered["topic_id"].value_counts().sort_values(ascending=False).values
    sorted_sizes = np.sort(topic_sizes)
    ccdf = 1.0 - np.arange(1, len(sorted_sizes) + 1) / len(sorted_sizes)

    df_ = pd.DataFrame({
        "sorted_sizes": sorted_sizes ,
        "ccdf": ccdf,
        "embedding": embedding_model_name,
        "model": "BERTopic"
    })
    df_.to_parquet(path_model_dir / f'CCDF_BERTopic_{embedding_model_name}.parquet')

    fig, ax = plt.subplots(figsize=(6, 4))  # white figure background

    ax.step(sorted_sizes, ccdf, where="post")
    ax.set_xlabel("Topic Size (Number of Documents)")
    ax.set_ylabel("P(Topic ≥ Size)")
    ax.set_title("Topic Size CCDF")
    ax.grid(True)

    plt.tight_layout()
    fig.savefig(path_figures / "topic_size_ccdf.png", dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()

    # Save cohesive-only CSV and stats
    retained_df.to_csv(path_model_dir / "4_umap_coords_cohesive.csv", index=False)
    with open(path_model_dir / "4_umap_stats.json", "w") as f:
        json.dump({
            "cohesion_threshold": float(cohesion_threshold),
            "n_retained_topics": int(n_retained),
            "n_total_topics": int(n_total),
            "x_min": float(shared_umap_limits["x_min"]),
            "x_max": float(shared_umap_limits["x_max"]),
            "y_min": float(shared_umap_limits["y_min"]),
            "y_max": float(shared_umap_limits["y_max"])
        }, f, indent=2)

    print(f"✓ Plots and stats saved for: {embedding_model_name}")


=== Generating UMAP plots for: all-MiniLM-L6-v2 ===

=== Generating UMAP plots for: all-mpnet-base-v2 ===

=== Generating UMAP plots for: distiluse-base-multilingual-cased-v1 ===

=== Generating UMAP plots for: paraphrase-MiniLM-L6-v2 ===
✓ Plots and stats saved for: all-MiniLM-L6-v2
✓ Plots and stats saved for: all-mpnet-base-v2
✓ Plots and stats saved for: distiluse-base-multilingual-cased-v1
✓ Plots and stats saved for: paraphrase-MiniLM-L6-v2


In [7]:
# Keep models with at least 2 finite cohesion scores
valid_models_bertopic = {
    name: [float(v) for v in vals if np.isfinite(v)]
    for name, vals in all_model_densities_bertopic.items()
    if sum(np.isfinite(vals)) > 1
}

# Run Kruskal-Wallis only if there are at least 2 valid groups
if len(valid_models_bertopic) < 2:
    print("Not enough valid topic density groups to perform Kruskal-Wallis test.")
else:
    groups_bertopic = list(valid_models_bertopic.values())
    h_stat_bertopic, p_value_bertopic = kruskal(*groups_bertopic)

    print("\n=== Kruskal-Wallis Test on BERTopic Topic Densities Across Models ===")
    print(f"H-statistic: {h_stat_bertopic:.10f}")
    print(f"p-value:     {p_value_bertopic:.2e}")
    print(f"Tested {len(groups_bertopic)} models.")
    print("Interpretation: p < 0.05 suggests a statistically significant difference in topic cohesion across models.")


=== Kruskal-Wallis Test on BERTopic Topic Densities Across Models ===
H-statistic: 32.4257625016
p-value:     4.26e-07
Tested 4 models.
Interpretation: p < 0.05 suggests a statistically significant difference in topic cohesion across models.


In [8]:
def build_plot_sheet(sheet_type, suffix_all, suffix_cohesive, out_name):
    plot_pairs = []
    for model_name in embedding_model_names:
        norm_name = model_name.replace("/", "_")
        fig_dir = path_res_topic_mod_bertopic / norm_name / "figures"
        img_all = fig_dir / f"filtered_topics_umap_all_{suffix_all}.png"
        img_cohesive = fig_dir / f"filtered_topics_umap_cohesive_{suffix_cohesive}.png"
        if img_all.exists() and img_cohesive.exists():
            plot_pairs.append((model_name, img_all, img_cohesive))

    # Target: 2 rows x N columns (each model gets 2 images, side by side)
    n_models = len(plot_pairs)
    n_cols = n_models  # Two images per model — one row for each type
    fig_width = 3 * n_cols
    fig_height = 6  # two rows

    fig, axes = plt.subplots(2, n_cols, figsize=(fig_width, fig_height))
    axes = np.atleast_2d(axes)

    for col_idx, (model_name, img_all, img_cohesive) in enumerate(plot_pairs):
        threshold = model_percentile_thresholds_bertopic.get(model_name, float('nan'))

        for row_idx, img_path in enumerate([img_all, img_cohesive]):
            img = Image.open(img_path)
            ax = axes[row_idx][col_idx]
            ax.imshow(img)
            ax.axis("off")

            title = (
                f"{model_name}\nAll Topics UMAP" if row_idx == 0
                else f"{model_name}\nCohesive Topics UMAP"
            )
            ax.set_title(title, fontsize=10)

    # Shared annotation
    fig.text(
        0.5, 0,
        kruskal_text,
        ha="center",
        fontsize=10
    )

    plt.tight_layout(rect=[0, 0.04, 1, 1])
    out_path = path_res_topic_mod_bertopic / out_name
    plt.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.close()
    print(f"{sheet_type} sheet saved to: {out_path}")


# Run Kruskal-Wallis test for BERTopic
valid_models_bertopic = {
    name: [float(v) for v in vals if np.isfinite(v)]
    for name, vals in all_model_densities_bertopic.items()
    if len([v for v in vals if np.isfinite(v)]) > 1
}
groups_bertopic = list(valid_models_bertopic.values())

if len(groups_bertopic) >= 2:
    h_stat, p_value = kruskal(*groups_bertopic)

    if p_value < 0.01:
        interpretation = (
            "Strong evidence of a significant difference in topic compactness across embedding models."
        )
    elif p_value < 0.05:
        interpretation = (
            "Moderate evidence of a difference in topic compactness across embedding models."
        )
    else:
        interpretation = (
            "No statistically significant difference in topic compactness across embedding models."
        )

    kruskal_text = (
        f"Kruskal-Wallis H = {h_stat:.2f}, p = {p_value:.2e} — Testing whether topic cohesion varies across embedding models.\n"
        f"→ {interpretation}\n"
        "This suggests that the choice of embedding model may influence clustering quality, depending on statistical significance."
    )
else:
    kruskal_text = "Insufficient valid topic cohesion data to perform the Kruskal-Wallis test."

# --- Generate both plot sheets ---
build_plot_sheet(
    sheet_type="Global axes",
    suffix_all="fixed_axes",
    suffix_cohesive="fixed_axes",
    out_name="umap_plot_sheet_global.png"
)

build_plot_sheet(
    sheet_type="Local axes",
    suffix_all="local_axes",
    suffix_cohesive="local_axes",
    out_name="umap_plot_sheet_local.png"
)

Global axes sheet saved to: /Users/ap4320861gmail.com/Library/Mobile Documents/com~apple~CloudDocs/minor-project/results/topic-modeling/bertopic/result/umap_plot_sheet_global.png
Local axes sheet saved to: /Users/ap4320861gmail.com/Library/Mobile Documents/com~apple~CloudDocs/minor-project/results/topic-modeling/bertopic/result/umap_plot_sheet_local.png


In [9]:
from PIL import Image

ccdf_images = []
for model_name in embedding_model_names:
    norm_name = model_name.replace("/", "_")
    fig_path = path_res_topic_mod_bertopic / norm_name / "figures" / "topic_size_ccdf.png"
    if fig_path.exists():
        ccdf_images.append((model_name, fig_path))

if ccdf_images:
    n_cols = len(ccdf_images)
    fig_width = 4 * n_cols
    row_height = 3

    fig, axes = plt.subplots(1, n_cols, figsize=(fig_width, row_height), constrained_layout=True)
    fig.patch.set_facecolor("white")  # ensures white background
    axes = np.atleast_1d(axes)

    for i, (model_name, img_path) in enumerate(ccdf_images):
        img = Image.open(img_path)
        axes[i].imshow(img, interpolation="none")
        axes[i].axis("off")
        axes[i].set_title(f"{model_name}", fontsize=11, pad=8)

    ccdf_out_path = path_res_topic_mod_bertopic / "topic_size_ccdf_sheet.png"
    plt.savefig(ccdf_out_path, dpi=300, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close()

    print(f"✓ CCDF sheet saved to: {ccdf_out_path}")
else:
    print("No CCDF plots found for the embedding models.")


✓ CCDF sheet saved to: /Users/ap4320861gmail.com/Library/Mobile Documents/com~apple~CloudDocs/minor-project/results/topic-modeling/bertopic/result/topic_size_ccdf_sheet.png


In [10]:
# ----------------- C_V for *cohesive* topics, per model -------------------
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel
import numpy as np

cohesive_cv_summary = []

for embedding_model_name in embedding_model_names:
    norm_name = embedding_model_name.replace("/", "_")
    path_model_dir = path_res_topic_mod_bertopic / norm_name
    path_model_file = path_model_dir / "bertopic_model.pkl"
    path_cohesive = path_model_dir / "4_umap_coords_cohesive.csv"

    if not (path_model_file.exists() and path_cohesive.exists()):
        print(f"Skipping {embedding_model_name} – missing cohesive data or model.")
        continue

    # 1 ─ load model and cohesive doc/topic IDs
    topic_model = BERTopic.load(str(path_model_file))
    retained_df = pd.read_csv(path_cohesive)

    cohesive_topic_ids = set(retained_df["topic_id"].unique())
    cohesive_doc_ids = retained_df["document_id"].astype(int).unique().tolist()

    # 2 ─ tokenise those documents with the *same* analyser
    analyser = topic_model.vectorizer_model.build_analyzer()
    tokenised_docs = [analyser(text_series[i]) for i in cohesive_doc_ids]

    if not tokenised_docs:
        print(f"No docs for cohesive topics in {embedding_model_name}")
        continue

    # 3 ─ gensim dictionary + topic word lists
    dictionary = Dictionary(tokenised_docs)

    top_n = 10
    topic_words = [
        [w for w, _ in topic_model.get_topic(tid)[:top_n]]
        for tid in sorted(cohesive_topic_ids) if tid != -1
    ]

    # 4 ─ C_V coherence
    cm = CoherenceModel(
        topics=topic_words,
        texts=tokenised_docs,
        dictionary=dictionary,
        coherence="c_v"
    )
    per_topic_scores = cm.get_coherence_per_topic()
    cv_overall = float(np.mean(per_topic_scores)) if per_topic_scores else float("nan")

    cohesive_cv_summary.append({
        "embedding_model": embedding_model_name,
        "n_cohesive_topics": len(cohesive_topic_ids),
        "cv_overall": cv_overall
    })

    print(f"{embedding_model_name:34s}  "
          f"C_V (cohesive) = {cv_overall:6.4f}  "
          f"on {len(cohesive_topic_ids)} topics")

# 5 ─ save summary
df_cv = pd.DataFrame(cohesive_cv_summary)
out_path = path_res_topic_mod_bertopic / "cohesive_cv_summary.csv"
df_cv.to_csv(out_path, index=False)
print(f"\n✓ Cohesive-topic C_V scores saved to: {out_path}")


all-MiniLM-L6-v2                    C_V (cohesive) = 0.4566  on 51 topics
all-mpnet-base-v2                   C_V (cohesive) = 0.4935  on 50 topics
distiluse-base-multilingual-cased-v1  C_V (cohesive) = 0.4283  on 48 topics
paraphrase-MiniLM-L6-v2             C_V (cohesive) = 0.4423  on 50 topics

✓ Cohesive-topic C_V scores saved to: /Users/ap4320861gmail.com/Library/Mobile Documents/com~apple~CloudDocs/minor-project/results/topic-modeling/bertopic/result/cohesive_cv_summary.csv
